###Setting up Environment

In [2]:
!pip install sqlalchemy psycopg2-binary #sqlalchemy Python classes to DB, psycopg PGsql adapter
!pip install pyspark

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 27.3 MB/s eta 0:00:00


In [1]:
# !wget https://jdbc.postgresql.org/download/postgresql-42.5.0.jar #jdbc  Java Database connectivity  #downloaded in mounted gdrive

--2025-06-30 06:16:08--  https://jdbc.postgresql.org/download/postgresql-42.5.0.jar
Resolving jdbc.postgresql.org (jdbc.postgresql.org)... 72.32.157.228, 2001:4800:3e1:1::228
Connecting to jdbc.postgresql.org (jdbc.postgresql.org)|72.32.157.228|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1046274 (1022K) [application/java-archive]
Saving to: ‘postgresql-42.5.0.jar’

postgresql-42.5.0.j 100%[===================>]   1022K  --.-KB/s    in 0.09s   

2025-06-30 06:16:08 (10.6 MB/s) - ‘postgresql-42.5.0.jar’ saved [1046274/1046274]



In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PostgreSQL Connection with PySpark") \
    .config("spark.jars", "/content/drive/MyDrive/Colab Notebooks/postgresql-42.5.0.jar") \
    .getOrCreate()

In [2]:
url = "jdbc:postgresql://ep-withered-bush-a1qfuygo-pooler.ap-southeast-1.aws.neon.tech/neondb"
properties = {
    "user": "neondb_owner",
    "password": "npg_sjakELOn7Bp4",
    "driver": "org.postgresql.Driver",
    "ssl": "true",
    "sslfactory": "org.postgresql.ssl.NonValidatingFactory"
}

In [3]:
import psycopg2
conn = psycopg2.connect(
      dbname="neondb",
      user="neondb_owner",
      password="npg_sjakELOn7Bp4",
      host="ep-withered-bush-a1qfuygo-pooler.ap-southeast-1.aws.neon.tech",
      sslmode="require"
  )
cursor = conn.cursor()

##EXTRACTION

In [56]:
df = spark.read.option("multiLine", "true").json("/content/drive/MyDrive/Colab Notebooks/JSON MOCK DATA/1.json")

In [57]:
df.show()

+---------+--------------------+--------------------+-----+-------------------+-------+
|     date|               event|                  ip| time|          time_zone|user_id|
+---------+--------------------+--------------------+-----+-------------------+-------+
|7/14/2024|        Link clicked|85cb:1b80:8d97:ec...|10:21|         Asia/Tokyo|     20|
|2/28/2025|        Video played|3d99:914:b7c4:440...| 1:35|      Europe/Lisbon|     34|
|11/8/2024|        Image viewed|b523:6f99:6037:d6...| 9:39|       Asia/Jakarta|     39|
| 2/8/2025|       Page scrolled|5626:b1e9:6f3c:b3...| 5:04|Australia/Melbourne|     18|
|3/19/2025|        Link clicked|4efa:71c9:fece:37...| 7:34|      Europe/Moscow|     34|
|3/18/2025|        Video played|1ba0:9b37:6d4f:3e...| 8:33|      Europe/Warsaw|      7|
| 4/9/2025|       Page scrolled|8bd0:f032:b44:4c4...| 7:22|      Europe/Warsaw|     45|
| 6/9/2025|   Dropdown selected|386:5649:4b21:c2f...| 0:22|     Asia/Chongqing|     30|
| 7/5/2024|               Login|

##TRANSFORMATION

In [58]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [59]:
df.printSchema()

root
 |-- date: string (nullable = true)
 |-- event: string (nullable = true)
 |-- ip: string (nullable = true)
 |-- time: string (nullable = true)
 |-- time_zone: string (nullable = true)
 |-- user_id: long (nullable = true)



In [60]:
import re
def get_ip_version(ip):
    if re.match(r'^(\d{1,3}\.){3}\d{1,3}$', ip):
        return 4
    elif re.match(r'^([0-9a-fA-F]{0,4}:){2,7}[0-9a-fA-F]{0,4}$', ip):
        return 6
    else:
        return None  # Invalid IP

# Register UDF (User Defined Function)
get_ip_version_udf = udf(get_ip_version, IntegerType())

df = df.withColumn("ip_version", get_ip_version_udf(df["ip"]))
df.show()

+---------+--------------------+--------------------+-----+-------------------+-------+----------+
|     date|               event|                  ip| time|          time_zone|user_id|ip_version|
+---------+--------------------+--------------------+-----+-------------------+-------+----------+
|7/14/2024|        Link clicked|85cb:1b80:8d97:ec...|10:21|         Asia/Tokyo|     20|         6|
|2/28/2025|        Video played|3d99:914:b7c4:440...| 1:35|      Europe/Lisbon|     34|         6|
|11/8/2024|        Image viewed|b523:6f99:6037:d6...| 9:39|       Asia/Jakarta|     39|         6|
| 2/8/2025|       Page scrolled|5626:b1e9:6f3c:b3...| 5:04|Australia/Melbourne|     18|         6|
|3/19/2025|        Link clicked|4efa:71c9:fece:37...| 7:34|      Europe/Moscow|     34|         6|
|3/18/2025|        Video played|1ba0:9b37:6d4f:3e...| 8:33|      Europe/Warsaw|      7|         6|
| 4/9/2025|       Page scrolled|8bd0:f032:b44:4c4...| 7:22|      Europe/Warsaw|     45|         6|
| 6/9/2025

In [61]:
df = df.dropDuplicates(["user_id", "event", "date", "time", "time_zone", "ip"]) #dropping Duplicates

In [62]:
df = df.withColumn("event",
    lower(regexp_replace(col("event"), r"[^\w]+", "_"))
)
df.show()

+----------+--------------------+--------------------+-----+-----------------+-------+----------+
|      date|               event|                  ip| time|        time_zone|user_id|ip_version|
+----------+--------------------+--------------------+-----+-----------------+-------+----------+
| 2/28/2025|        video_played|3d99:914:b7c4:440...| 1:35|    Europe/Lisbon|     34|         6|
| 7/29/2024|radio_button_sele...|948e:f0b3:1df4:c2...| 3:00|     Asia/Bangkok|     26|         6|
|11/12/2024|               login|5a03:ac9b:f03b:b5...| 6:06|   America/Regina|     28|         6|
| 6/18/2025|        link_clicked|a17b:17d9:818b:11...| 3:05|   Asia/Jerusalem|     26|         6|
| 2/17/2025|    checkbox_checked|de7f:58a2:af9:678...|11:25|       Asia/Tokyo|     12|         6|
| 1/21/2025|      form_submitted|dc10:d576:e5bf:4b...| 0:38|      Asia/Manila|      4|         6|
| 3/18/2025|        video_played|1ba0:9b37:6d4f:3e...| 8:33|    Europe/Warsaw|      7|         6|
|  7/5/2024|        

In [63]:
import datetime
import pytz
def to_utc(date_str, time_str, tz_str):
    try:
        naive = datetime.datetime.strptime(f"{date_str} {time_str}", "%m/%d/%Y %H:%M")
        local_tz = pytz.timezone(tz_str)
        local_dt = local_tz.localize(naive, is_dst=None)
        utc_dt = local_dt.astimezone(pytz.utc)
        return utc_dt.strftime("%Y-%m-%dT%H:%M:%S")
    except Exception:
        return None

to_utc_udf = udf(to_utc, StringType())
df = df.withColumn("utc_timestamp", to_utc_udf(col("date"), col("time"), col("time_zone")))
df.show()

+----------+--------------------+--------------------+-----+-----------------+-------+----------+-------------------+
|      date|               event|                  ip| time|        time_zone|user_id|ip_version|      utc_timestamp|
+----------+--------------------+--------------------+-----+-----------------+-------+----------+-------------------+
| 2/28/2025|        video_played|3d99:914:b7c4:440...| 1:35|    Europe/Lisbon|     34|         6|2025-02-28T01:35:00|
| 7/29/2024|radio_button_sele...|948e:f0b3:1df4:c2...| 3:00|     Asia/Bangkok|     26|         6|2024-07-28T20:00:00|
|11/12/2024|               login|5a03:ac9b:f03b:b5...| 6:06|   America/Regina|     28|         6|2024-11-12T12:06:00|
| 6/18/2025|        link_clicked|a17b:17d9:818b:11...| 3:05|   Asia/Jerusalem|     26|         6|2025-06-18T00:05:00|
| 2/17/2025|    checkbox_checked|de7f:58a2:af9:678...|11:25|       Asia/Tokyo|     12|         6|2025-02-17T02:25:00|
| 1/21/2025|      form_submitted|dc10:d576:e5bf:4b...| 0

In [64]:
df = df.withColumn("hour_of_day", hour(to_timestamp("utc_timestamp")))
df = df.withColumn("weekday", date_format(to_timestamp("utc_timestamp"), "EEEE"))
df = df.withColumn("region", split(col("time_zone"), "/").getItem(0))
df.show()

+----------+--------------------+--------------------+-----+-----------------+-------+----------+-------------------+-----------+---------+-------+
|      date|               event|                  ip| time|        time_zone|user_id|ip_version|      utc_timestamp|hour_of_day|  weekday| region|
+----------+--------------------+--------------------+-----+-----------------+-------+----------+-------------------+-----------+---------+-------+
| 2/28/2025|        video_played|3d99:914:b7c4:440...| 1:35|    Europe/Lisbon|     34|         6|2025-02-28T01:35:00|          1|   Friday| Europe|
| 7/29/2024|radio_button_sele...|948e:f0b3:1df4:c2...| 3:00|     Asia/Bangkok|     26|         6|2024-07-28T20:00:00|         20|   Sunday|   Asia|
|11/12/2024|               login|5a03:ac9b:f03b:b5...| 6:06|   America/Regina|     28|         6|2024-11-12T12:06:00|         12|  Tuesday|America|
| 6/18/2025|        link_clicked|a17b:17d9:818b:11...| 3:05|   Asia/Jerusalem|     26|         6|2025-06-18T00:0

In [65]:
df = df.filter(col("user_id").isNotNull() & col("utc_timestamp").isNotNull())
df.show()

+----------+--------------------+--------------------+-----+-----------------+-------+----------+-------------------+-----------+---------+-------+
|      date|               event|                  ip| time|        time_zone|user_id|ip_version|      utc_timestamp|hour_of_day|  weekday| region|
+----------+--------------------+--------------------+-----+-----------------+-------+----------+-------------------+-----------+---------+-------+
| 2/28/2025|        video_played|3d99:914:b7c4:440...| 1:35|    Europe/Lisbon|     34|         6|2025-02-28T01:35:00|          1|   Friday| Europe|
| 7/29/2024|radio_button_sele...|948e:f0b3:1df4:c2...| 3:00|     Asia/Bangkok|     26|         6|2024-07-28T20:00:00|         20|   Sunday|   Asia|
|11/12/2024|               login|5a03:ac9b:f03b:b5...| 6:06|   America/Regina|     28|         6|2024-11-12T12:06:00|         12|  Tuesday|America|
| 6/18/2025|        link_clicked|a17b:17d9:818b:11...| 3:05|   Asia/Jerusalem|     26|         6|2025-06-18T00:0

##LOADING

In [77]:
try:
  df.write.jdbc(url=url, table='"qa".events', mode="append", properties=properties)
except:
  print("UNIQUE KEY VIOLATION/INVALID DATA!")

UNIQUE KEY VIOLATION/INVALID DATA!
